In [1]:
import pandas as pd

# Single CSV exploration: build_dataset.py and download.py development

In [2]:
# Load sin CSV
df = pd.read_csv("../data/raw/job-bank-open-data-all-job-postings-en-may2026.csv", encoding="utf-16", sep="\t", low_memory=False)

In [3]:

# Normalize column names: collapse multiple spaces, strip leading/trailing
df.columns = df.columns.str.strip().str.replace(r"\s+", " ", regex=True)

missing_city = df[df["City"].isna()]

In [4]:
print("Rows missing City:", len(missing_city))
print()
print("Their Province/Territory values:")
print(missing_city["Province/Territory"].value_counts())
print()
print("Their Economic Region values:")
print(missing_city["Economic Region"].value_counts())

Rows missing City: 465

Their Province/Territory values:
Province/Territory
Québec                       137
Ontario                      133
British Columbia              85
Alberta                       31
Nova Scotia                   31
Saskatchewan                  26
New Brunswick                  9
Manitoba                       7
Newfoundland and Labrador      2
Nunavut                        2
Prince Edward Island           2
Name: count, dtype: int64

Their Economic Region values:
Series([], Name: count, dtype: int64)


In [5]:
print("Also missing Economic Region:", missing_city["Economic Region"].isna().sum(), "/", len(missing_city))
print()
print("Postal code populated for these rows:", missing_city["Work Location Postal Code"].notna().sum(), "/", len(missing_city))
print()
print("Sample postal codes (first 3 chars = FSA):")
print(missing_city["Work Location Postal Code"].dropna().str[:3].value_counts().head(15))

Also missing Economic Region: 465 / 465

Postal code populated for these rows: 0 / 465

Sample postal codes (first 3 chars = FSA):
Series([], Name: count, dtype: int64)


# Final dataset revision

In [6]:
df1 = pd.read_csv("../data/processed/vancouver_tech_postings.csv", low_memory=False, parse_dates=["First Posting Date"])

In [7]:
df1

,WIC Job Location Snapshot ID,Job Title,Original Job Title,NOC 2016 Code,NOC 2016 Code Name,NOC21 Code,NOC21 Code Name,External Indicator,First Posting Date,Vacancy Count,...,Commission PER,Commission Type,Hours Per,Hours Minimum,Hours Maximum,Work Hours,Work Hours From Time,Work Hours To Time,Salary Min Annual,Salary Max Annual
0,15656146.0,information technology (IT) analyst,Lighting Specialist,2171.0,Information systems analysts and consultants,21222.0,Information systems specialists,1,2026-06-19,1,...,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN,95000.0,95000.0
1,15668830.0,technical support engineer,NaN,2171.0,Information systems analysts and consultants,21222.0,Information systems specialists,1,2026-06-26,1,...,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN,72000.0,85000.0
2,15606233.0,software developer,Mobile Security Professaional - Healthcare Sec...,2174.0,Computer programmers and interactive media dev...,21232.0,Software developers and programmers,1,2026-06-03,1,...,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN,50700.0,50700.0
3,15612219.0,information technology (IT) analyst,Inside Sales Specialist,2171.0,Information systems analysts and consultants,21222.0,Information systems specialists,1,2026-06-04,1,...,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN,54600.0,68250.0
4,15616226.0,Web developer,Backend Developer ( Nest.js Typescript),2175.0,Web designers and developers,21234.0,Web developers and programmers,1,2026-06-05,1,...,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN,80000.0,120000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4100,NaN,cloud operations manager,Operations Manager,213.0,Computer and information systems managers,20012.0,Computer and information systems managers,1,2024-07-30,1,...,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN,NaN,NaN
4101,NaN,"manager, information systems",D365 Technical Project Manager,213.0,Computer and information systems managers,20012.0,Computer and information systems managers,1,2024-07-31,1,...,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN,NaN,NaN
4102,NaN,informatics consultant,"Eyecare Consultant - West Vancouver, BC",2171.0,Information systems analysts and consultants,21222.0,Information systems specialists,1,2024-07-01,1,...,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN,NaN,NaN
4103,NaN,QA (quality assurance) analyst - informatics,Quality Assurance Technician,2171.0,Information systems analysts and consultants,21222.0,Information systems specialists,1,2024-07-02,1,...,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN,NaN,NaN


In [8]:
df1['City'].unique()

<StringArray>
[        'Burnaby',       'Vancouver',        'Richmond',          'Surrey',
 'North Vancouver',       'Coquitlam',         'Langley',      'White Rock',
           'Delta',  'West Vancouver', 'New Westminster',  'Port Coquitlam',
     'Maple Ridge',    'Pitt Meadows',      'Port Moody']
Length: 15, dtype: str

In [9]:
print(df1[["Salary Minimum", "Salary Maximum"]].notna().sum())
print(f"Total rows: {len(df1)}")

Salary Minimum    2824
Salary Maximum    2824
dtype: int64
Total rows: 4105


In [10]:
# How wide is Salary Minimum vs Maximum — some postings report ranges, others identical min/max
print((df1["Salary Maximum"] - df1["Salary Minimum"]).describe())

count      2824.000000
mean       5991.455124
std       14605.205516
min           0.000000
25%           0.000000
50%           0.000000
75%          51.160000
max      153050.000000
dtype: float64


In [11]:
# Sanity check for garbage values (e.g. $0, or suspiciously low/high salaries)
print(df1[["Salary Minimum", "Salary Maximum"]].describe())

       Salary Minimum  Salary Maximum
count     2824.000000     2824.000000
mean     34666.263612    40657.718736
std      47945.182357    56593.388318
min         10.000000       10.000000
25%         38.500000       41.000000
50%         55.000000       60.860000
75%      72000.000000    87775.000000
max     250000.000000   323050.000000


In [12]:
# Salary Per matters — hourly vs annual figures mixed together will wreck a model
print(df1["Salary Per"].value_counts())

Salary Per
Hour         1620
Year         1076
Month          77
Bi-weekly      50
Week            1
Name: count, dtype: int64


In [13]:
print(df1["First Posting Date"].min(), "to", df1["First Posting Date"].max())
print(df1["First Posting Date"].dt.to_period("M").nunique(), "unique months")

2024-07-01 00:00:00 to 2026-06-30 00:00:00
24 unique months


# Notes and Information: Raw dataset and final Metro Vancouver dataset

* Raw rows loaded: 1,783,070
* Note: 1,411 tech-NOC postings have no City (and no Economic Region / postal code fallback available) — excluded since they can't be confirmed as Metro Vancouver.
* Note: 597 rows matched a Metro Vancouver city name but were NOT in British Columbia — excluded. Examples:


| Number of data points | City       | Province/Territory   |
|-----------------------|------------|-----------------------|
| 366                    | Richmond   | Québec                |
| 4032                   | Richmond   | Ontario               |
| 4386                   | Delta      | Ontario               |
| 184770                 | White Rock | Nova Scotia           |
| 467368                 | Richmond   | Prince Edward Island  |

* Vancouver tech postings after filtering: 4,379